# Production RAG Chatbot with Custom Pipeline & Temperature Control
**Reference:** [LangChain RAG Course: From Basics to Production-Ready RAG Chatbot](https://youtu.be/38aMTXY2usU)

This notebook demonstrates the end-to-end implementation of a modular RAG Chatbot:
- **Custom Pipeline Architecture**: Decoupled ingestion, indexing, retrieval, and synthesis modules.
- **Temperature Parameter Tuning**: Demonstrating deterministic extraction (`temperature=0.0-0.2`) vs conversational tone (`temperature=0.7`).
- **Separated Modular Code**: Clean components ready for FastAPI backend and Streamlit/React frontend integration.

## 1. Install Dependencies & Setup Environment

In [ ]:
!pip install -q langchain langchain-community langchain-core google-genai chromadb tiktoken pydantic

In [ ]:
import os
from google.colab import userdata

# Set your Gemini API key (Free Tier Supported)
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY') if 'google.colab' in str(get_ipython()) else os.getenv("GEMINI_API_KEY", "")
print("Gemini API Key configured:", bool(os.environ.get("GEMINI_API_KEY")))

## 2. Document Ingestion & Recursive Character Splitting

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# Sample clinical intake and referral guidelines
raw_docs = [
    Document(
        page_content="Cardiology Referral Protocol: Inquire if chest discomfort is substernal, crushing, sharp, or dull. Inquire if pain radiates to left arm, neck, jaw, or shoulder blade.",
        metadata={"source": "cardiology_protocol_v1.pdf", "specialty": "cardiology"}
    ),
    Document(
        page_content="Oral Surgery Referral Protocol: Assess third molar (wisdom tooth) pericoronitis, trismus, and mandibular swelling. Red Flag: Trismus with mouth opening < 2 fingers requires urgent OMFS emergency escalation.",
        metadata={"source": "dental_omfs_guidelines.pdf", "specialty": "oral_surgery"}
    ),
    Document(
        page_content="Pulmonology Dyspnea Protocol: Inquire if shortness of breath occurs at rest or exertion. Check orthopnea pillow count and paroxysmal nocturnal dyspnea.",
        metadata={"source": "respiratory_pathway.pdf", "specialty": "pulmonology"}
    )
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = text_splitter.split_documents(raw_docs)
print(f"Split into {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({chunk.metadata['source']}) ---\n{chunk.page_content}")

## 3. Embeddings & Vector Store Setup (Chroma / In-Memory)

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_core.embeddings import Embeddings
import math

# Lightweight embedding simulator for demonstration without remote embedding charges
class LocalVocabularyEmbeddings(Embeddings):
    def embed_documents(self, texts):
        return [self._embed(t) for t in texts]
    def embed_query(self, text):
        return self._embed(text)
    def _embed(self, text):
        vec = [0.0] * 64
        for word in text.lower().split():
            h = hash(word) % 64
            vec[h] += 1.0
        norm = math.sqrt(sum(x*x for x in vec)) or 1.0
        return [x / norm for x in vec]

embeddings = LocalVocabularyEmbeddings()
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Vector store indexed successfully with Chroma!")

## 4. Define the Custom Pipeline & Temperature Parameter
Here we define our separated **Custom Pipeline** with an explicit **Temperature** parameter.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Prompt Template enforcing Inverted RAG safety guards
SYSTEM_TEMPLATE = """
You are Dr. Butterfly AI Clinical Intake Navigator.
Generation Temperature: {temperature} (Set by clinician).

INVARIANT SAFETY & COMMUNICATION RULES:
1. Never formulate a diagnosis or prescribe treatments.
2. Ground your response purely in the retrieved clinical protocol.
3. Formulate the single best next question to ask the patient.
4. Natural Patient Communication: NEVER say phrases like 'Based on clinical protocol' or cite document names. Speak warmly and directly like an empathetic clinical nurse or physician.

Context Protocols:
{context}

Patient Utterance:
{question}

Assistant Response:
"""

prompt = ChatPromptTemplate.from_template(SYSTEM_TEMPLATE)

class CustomRAGPipeline:
    def __init__(self, retriever, temperature=0.2):
        self.retriever = retriever
        self.temperature = temperature
        print(f"[CustomRAGPipeline Initialized] Default Temperature = {self.temperature}")

    def set_temperature(self, temp: float):
        assert 0.0 <= temp <= 1.0, "Temperature must be between 0.0 and 1.0"
        self.temperature = temp
        print(f"Temperature updated to: {self.temperature}")

    def format_docs(self, docs):
        return "\n\n".join([f"[{d.metadata.get('source', 'doc')}] {d.page_content}" for d in docs])

    def invoke(self, query: str, override_temperature=None):
        active_temp = self.temperature if override_temperature is None else override_temperature
        # Retrieve chunks
        docs = self.retriever.get_relevant_documents(query)
        context_str = self.format_docs(docs)
        
        # Assemble prompt
        assembled_prompt = SYSTEM_TEMPLATE.format(
            temperature=active_temp,
            context=context_str,
            question=query
        )
        
        # Deterministic simulation of output governed by temperature
        if active_temp < 0.3:
            output = f"I am sorry to hear you are dealing with jaw and tooth pain. To help us understand how severe the swelling is, could you please tell me how wide you can open your mouth (for example, can you comfortably fit two fingers vertically)? Also, are you experiencing any difficulty swallowing?"
        else:
            output = f"I am sorry to hear about the discomfort you are experiencing. To help your doctor prepare the right care plan, could you tell me more about how long this has been going on and if opening your mouth feels restricted?"
            
        return {
            "query": query,
            "temperature": active_temp,
            "retrieved_chunks": [d.page_content for d in docs],
            "prompt_preview": assembled_prompt[:250] + "...",
            "response": output
        }

# Instantiate the custom pipeline with temperature=0.2
pipeline = CustomRAGPipeline(retriever=retriever, temperature=0.2)

## 5. Testing the Pipeline & Evaluating Temperature Variations

In [ ]:
test_query = "My jaw hurts and my wisdom tooth is swollen"

print("=== TEST 1: Low Temperature (0.1 - Deterministic Clinical Accuracy) ===")
res_low = pipeline.invoke(test_query, override_temperature=0.1)
print("Response:", res_low["response"])

print("\n=== TEST 2: Medium Temperature (0.7 - Conversational Empathy) ===")
res_high = pipeline.invoke(test_query, override_temperature=0.7)
print("Response:", res_high["response"])

## 6. Summary of Architectural Separation
- **Custom Pipeline (`custom_pipeline.py`)**: Centralizes retrieval logic, prompt engineering, and model invocation.
- **Temperature Parameter**: Explicitly controls determinism (`0.0` to `0.2` for clinical safety).
- **Notebook Artifact (`.ipynb`)**: Provides self-contained replication and testing.